# Clase 10: Bases de datos NoSQL

**Objetivos de aprendizaje:**
- Explicar por qué existe NoSQL.
- Diferenciar los cuatro tipos (clave-valor, documentos, columnas, grafos) y su caso de uso típico.
- Representar un documento NoSQL como estructura Python/JSON.

**Duración estimada:** 110 min


## ¿Por qué necesitamos algo más que SQL?

En la clase anterior vimos que las bases de datos relacionales son extraordinariamente fiables gracias a ACID. Pero tienen un punto débil: exigen un esquema fijo, y no siempre escalan bien cuando el volumen de lecturas y escrituras es descomunal (piensa en una red social con cientos de millones de usuarios escribiendo a la vez).

**NoSQL** ("Not Only SQL") es la familia de bases de datos que renuncian a parte de esa rigidez a cambio de flexibilidad y escalado masivo. El impulso histórico vino de un whitepaper de Google de 2006 sobre un sistema llamado BigTable, que inspiró alternativas como Cassandra o HBase.

No hay "una" base de datos NoSQL: hay **cuatro tipos distintos**. Vamos a simular cada uno con Python puro para entender la idea de fondo de cada uno.

In [ ]:
# 1. CLAVE-VALOR (como Redis, Memcached, DynamoDB)
# La estructura más simple posible: cada clave apunta a un valor,
# sin que el sistema sepa (ni le importe) qué hay dentro del valor.
sesiones_activas = {
    "usuario_284": "activo_desde_10:32",
    "usuario_591": "activo_desde_10:47",
    "usuario_112": "activo_desde_11:03",
}

print("Sesión del usuario_591:", sesiones_activas["usuario_591"])
print("\nCaso de uso típico: sesiones, caché, recomendaciones rápidas.")

## Documentos: cuando cada registro tiene su propia forma

En una tabla SQL, todas las filas comparten exactamente las mismas columnas. En una base **basada en documentos** (MongoDB, CouchDB, Cloudant...), cada "documento" puede tener una estructura distinta — ideal para catálogos de eCommerce, donde una camiseta tiene talla y color, y unos auriculares tienen autonomía y bluetooth.

In [ ]:
import json

productos = [
    {
        "id_producto": "P001",
        "nombre": "Camiseta básica",
        "categoria": "ropa",
        "atributos": {"talla": "M", "color": "azul"},
        "precio": 15.99,
    },
    {
        "id_producto": "P002",
        "nombre": "Auriculares inalámbricos",
        "categoria": "electrónica",
        "atributos": {"autonomia_horas": 20, "bluetooth": True},
        "precio": 39.90,
    },
]

# json.dumps muestra el documento tal como se guardaría en Mongo
print(json.dumps(productos[0], indent=2, ensure_ascii=False))

def buscar_por_categoria(coleccion, categoria):
    return [doc for doc in coleccion if doc["categoria"] == categoria]

print("\nProductos de electrónica:")
print(buscar_por_categoria(productos, "electrónica"))

## Columnar: guardar por columna en vez de por fila

Las bases **columnares** (Cassandra, HBase) agrupan los valores de una misma columna juntos en memoria, en vez de agrupar por fila como hace SQL. Esto las hace extraordinariamente rápidas para escrituras masivas y para leer una sola columna sin tocar las demás — el caso típico son series temporales de sensores IoT.

In [ ]:
# En vez de una lista de filas, una lista por columna.
lecturas_sensores = {
    "id_sensor":    ["S1", "S2", "S3", "S4"],
    "temperatura":  [21.3, 19.8, 22.1, 20.5],
    "humedad":      [55, 60, 48, 58],
}

print("Todas las temperaturas de golpe, sin tocar otras columnas:")
print(lecturas_sensores["temperatura"])

## Grafos: cuando lo importante es la relación

Las bases de datos de **grafos** (Neo4j, CosmosDB) representan los datos como nodos conectados por relaciones. Son el modelo natural para redes sociales, detección de fraude y sistemas de recomendación — preguntas del tipo "¿quiénes son los amigos de mis amigos?".

In [ ]:
red_amistades = {
    "Ana":   ["Luis", "Carla"],
    "Luis":  ["Ana", "Diego"],
    "Carla": ["Ana", "Marta"],
    "Diego": ["Luis"],
    "Marta": ["Carla"],
}

# BFS (recorrido en anchura): primero amigos directos (nivel 1),
# luego amigos de esos amigos (nivel 2), sin repetir a nadie.
def amigos_de_amigos(grafo, persona):
    visitados = {persona}
    cola = [(vecino, 1) for vecino in grafo.get(persona, [])]
    resultado_nivel_2 = []
    while cola:
        actual, nivel = cola.pop(0)
        if actual in visitados:
            continue
        visitados.add(actual)
        if nivel == 2:
            resultado_nivel_2.append(actual)
            continue
        for vecino in grafo.get(actual, []):
            if vecino not in visitados:
                cola.append((vecino, nivel + 1))
    return resultado_nivel_2

print("Amigos de amigos de Ana:", amigos_de_amigos(red_amistades, "Ana"))

## RDBMS vs. NoSQL: la tabla que hay que tener clara

| | RDBMS | NoSQL |
|---|---|---|
| Esquema | Rígido | Flexible |
| Garantías | ACID | Prioriza escalabilidad |
| Madurez | Muy madura | Más nueva |
| Hardware | Suele requerir máquinas potentes | Escala barato con hardware básico |

La elección real depende del caso de uso, tipo de dato, rendimiento, volumen y estándares de la organización. No es raro combinar varias soluciones a la vez dentro de la misma empresa.

## Resumen de la clase

- NoSQL existe porque las RDBMS no encajan bien con todos los tipos de datos ni todas las escalas.
- Cuatro tipos: **clave-valor** (dict simple), **documentos** (lista de dicts anidados), **columnar** (dict de listas), **grafos** (dict de adyacencia).
- Cada tipo tiene su caso de uso ideal: caché/sesiones, eCommerce/CRM, series temporales/IoT, redes sociales/fraude.
- Podemos simular fielmente los cuatro tipos con estructuras nativas de Python.

**Capítulo del manual:** [12 — Bases de datos NoSQL](../manual/12_bases_nosql.md)

**Notebook de práctica:** [clase10_practica_bases_nosql.ipynb](../notebooks_practica/clase10_practica_bases_nosql.ipynb)